# Pesquisa Informada — Greedy e A*

Neste notebook são apresentados e avaliados dois algoritmos de pesquisa informada:

- **Greedy Best-First Search**
- **A* Search**

Os algoritmos serão aplicados ao problema de recolha e entrega de produtos locais, utilizando os datasets `tiny` e `small`.

O objetivo é encontrar rotas que respeitem as restrições do problema e minimizar a distância total percorrida.

In [1]:
import sys
from pathlib import Path

# Descobrir a pasta principal do projeto
current = Path.cwd()

if (current / "src").is_dir():
    project_root = current
elif (current.parent / "src").is_dir():
    project_root = current.parent
else:
    raise FileNotFoundError(
        "Não encontrei a pasta src. "
        "Confirma que abriste o projeto correto."
    )

# Permitir importar ficheiros da pasta src
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Projeto:", project_root)

Projeto: /Users/davidfaria/Documents/IPCA 26_27/IA/IA2026_TP_G05


In [2]:
import importlib
import src.greedy

importlib.reload(src.greedy)

from src.greedy import (
    greedy_search,
    heuristic01,
    heuristic02
)

print("Greedy atualizado!")

Greedy atualizado!


In [6]:
from src.model import Problem, check_solution
from src.greedy import greedy_search, heuristic01, heuristic02
from datetime import datetime
from time import perf_counter

# ALTERAR AQUI: 4, 5, 6, 10 ou 12
num_orders = 12

# Carregar o Tiny completo
problem_tiny = Problem.from_csv(
    str(project_root / "data_tiny")
)

# Selecionar as primeiras N encomendas
order_ids = list(dict.fromkeys(
    r.order_id for r in problem_tiny.requests
))[:num_orders]

selected_requests = [
    r for r in problem_tiny.requests
    if r.order_id in order_ids
]

# Construir o problema de teste
problem_test = Problem(
    selected_requests,
    problem_tiny.vehicles,
    problem_tiny.distances,
    problem_tiny.durations,
    problem_tiny.warehouse_location,
    problem_tiny.rules
)

start_time = datetime(2026, 10, 30)

print("Encomendas:", order_ids)
print("Produtos:", len(problem_test.requests))
print()

# Testar as heurísticas
for heuristic in [heuristic01, heuristic02]:

    start = perf_counter()

    solution, expanded, complete = greedy_search(
        problem_test,
        start_time,
        heuristic,
        max_expansions=100000
    )

    elapsed = perf_counter() - start

    valid, distance, message = check_solution(
        problem_test,
        solution,
        start_time
    )

    print("Heurística:", heuristic.__name__)
    print("Completa:", complete)
    print("Válida:", valid)
    print("Distância:", distance, "km")
    print("Nós expandidos:", expanded)
    print("Tempo:", round(elapsed, 3), "s")
    print("Mensagem:", message)
    print()

Encomendas: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Produtos: 40

Heurística: heuristic01
Completa: True
Válida: True
Distância: 805.65 km
Nós expandidos: 449
Tempo: 0.138 s
Mensagem: Solução válida

Heurística: heuristic02
Completa: True
Válida: True
Distância: 762.77 km
Nós expandidos: 59654
Tempo: 14.618 s
Mensagem: Solução válida



In [7]:
from src.model import Problem, check_solution
from src.greedy import greedy_search, heuristic01, heuristic02
from datetime import datetime
from time import perf_counter

# Carregar o dataset Small completo
problem_small = Problem.from_csv(
    str(project_root / "data_small")
)

start_time = datetime(2026, 10, 30)

print("Dataset: Small")
print("Encomendas:", len(set(r.order_id for r in problem_small.requests)))
print("Produtos:", len(problem_small.requests))
print("Veículos:", len(problem_small.vehicles))
print()

# Testar as duas heurísticas
for heuristic in [heuristic01, heuristic02]:

    print("A testar:", heuristic.__name__, flush=True)

    inicio = perf_counter()

    solution, expanded, complete = greedy_search(
        problem_small,
        start_time,
        heuristic,
        max_expansions=100000
    )

    tempo = perf_counter() - inicio

    valid, distance, message = check_solution(
        problem_small,
        solution,
        start_time
    )

    print("Completa:", complete)
    print("Válida:", valid)
    print("Distância:", round(distance, 2), "km")
    print("Nós expandidos:", expanded)
    print("Tempo:", round(tempo, 3), "s")
    print("Mensagem:", message)
    print("-" * 40)

Dataset: Small
Encomendas: 25
Produtos: 80
Veículos: 2

A testar: heuristic01
Completa: False
Válida: False
Distância: 0.0 km
Nós expandidos: 100000
Tempo: 57.194 s
Mensagem: Encomendas por entregar
----------------------------------------
A testar: heuristic02
Completa: False
Válida: False
Distância: 0.0 km
Nós expandidos: 100000
Tempo: 113.455 s
Mensagem: Encomendas por entregar
----------------------------------------
